# WordCloud Data Preprocessing
This notebook processes Data/v2_final.tsv into appropriate description JSON chunks for D3 WordCloud visualization.

In [1]:
import numpy as np
import pandas as pd
import json
import os

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [2]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/andrewturangan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/andrewturangan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### Load TSV

In [3]:
tsvpath = os.path.join("..", "Data", "v2_final.tsv")
df = pd.read_csv(tsvpath, sep='\t')

/var/folders/yh/z83nv3dd2hl49zy2b2zxjx4h0000gn/T/ipykernel_89547/1396655357.py:2: DtypeWarning: Columns (63) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(tsvpath, sep='\t')


In [4]:
df = df.set_index('index')
df.head()

,city,country,description,location,state,state_abbrev,longitude,latitude,city_longitude,city_latitude,...,Optional_LATITUDE3,Optional_LATITUDE4,Optional_LONGITUDE1,Optional_LONGITUDE2,Optional_LONGITUDE3,Optional_LONGITUDE4,Optional_NAME1,Optional_NAME2,Optional_NAME3,Optional_NAME4
index,,,,,,,,,,,,,,,,,,,,,
0,Ada,United States,Ada witch - Sometimes you can see a misty blue...,Ada Cemetery,Michigan,MI,-85.504893,42.962106,-85.495480,42.960727,...,NaN,NaN,-85.50056,-85.49169,NaN,NaN,Egypt Valley Country Club,Findlay Cemetery,NaN,NaN
1,Addison,United States,A little girl was killed suddenly while waitin...,North Adams Rd.,Michigan,MI,-84.381843,41.971425,-84.347168,41.986434,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Adrian,United States,If you take Gorman Rd. west towards Sand Creek...,Ghost Trestle,Michigan,MI,-84.035656,41.904538,-84.037166,41.897547,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Adrian,United States,"In the 1970's, one room, room 211, in the old ...",Siena Heights University,Michigan,MI,-84.017565,41.905712,-84.037166,41.897547,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Albion,United States,Kappa Delta Sorority - The Kappa Delta Sororit...,Albion College,Michigan,MI,-84.745177,42.244006,-84.753030,42.243097,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Prepare decriptions without stopwords

In [5]:
stop_words = set(stopwords.words("english"))

In [6]:
def clean_description(text):
    words = word_tokenize(text)
    filtered = [word for word in words if word.lower() not in stop_words]
    return " ".join(filtered)

In [7]:
descriptions = df["description"].dropna().apply(clean_description).tolist()

In [13]:
subsets = {}
for i in range(df.shape[0]):
    subset = i % 50
    if subset + 1 not in subsets:
        subsets[subset + 1] = [descriptions[i]]
        continue
    subsets[subset + 1].append(descriptions[i])

### Generate JSONs

In [16]:
for i in range(1, 51):
    outfile = os.path.join("react-ui", "public", "WordCloud", f"descriptions_{i}.json")
    with open(outfile, "w", encoding="utf-8") as f:
        json.dump(subsets[i], f, indent=2, ensure_ascii=False)